## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify drive is mounted
!ls /content/drive/MyDrive/

## 2. Install Dependencies

**Note:** This cell installs the required AI/ML libraries.

In [ ]:
# Install required packages
print("📦 Installing packages (this may take 2-3 minutes)...")

!pip install -q insightface onnxruntime deepface opencv-python-headless tensorflow

print("✅ All packages installed!")

## 3. Import Libraries

In [ ]:
import os
import json
import time
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

# DeepFace for face detection AND emotion detection
from deepface import DeepFace

print("✅ Libraries imported successfully")

## 4. Configuration

**MODIFY THIS CELL:** Set the job_id from your web application.

In [ ]:
# ========================================
# CONFIGURATION - MODIFY THIS!
# ========================================

# 🔴 IMPORTANT: Get job_id from your web application (http://localhost:3000)
# 1. Upload a video on the web app
# 2. Copy the job_id from the response
# 3. Paste it below, replacing "YOUR_JOB_ID_HERE"
# Example: "123e4567-e89b-12d3-a456-426614174000"

JOB_ID = "YOUR_JOB_ID_HERE"  # ← CHANGE THIS TO YOUR JOB ID!

# Google Drive base path
DRIVE_BASE = "/content/drive/MyDrive/ai_emotion_project"

# Paths
INPUT_VIDEO_PATH = f"{DRIVE_BASE}/input_videos/{JOB_ID}.mp4"
OUTPUT_VIDEO_PATH = f"{DRIVE_BASE}/output_videos/{JOB_ID}.mp4"
OUTPUT_JSON_PATH = f"{DRIVE_BASE}/output_json/{JOB_ID}.json"
STATUS_FILE_PATH = f"{DRIVE_BASE}/status/{JOB_ID}.json"

# Ensure folders exist
os.makedirs(f"{DRIVE_BASE}/input_videos", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/output_videos", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/output_json", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/status", exist_ok=True)

print(f"✅ Configuration set for job: {JOB_ID}")
print(f"📹 Input video: {INPUT_VIDEO_PATH}")
print(f"📹 Output video: {OUTPUT_VIDEO_PATH}")
print(f"📊 Output JSON: {OUTPUT_JSON_PATH}")
print(f"📝 Status file: {STATUS_FILE_PATH}")

## 5. Helper Functions

In [ ]:
# ========================================
# HELPER FUNCTIONS
# ========================================

import numpy as np
from collections import defaultdict

def update_status(status, message, progress=None):
    """Update the status file for the web application to read."""
    status_data = {
        "status": status,
        "message": message,
        "timestamp": time.time()
    }
    if progress is not None:
        status_data["progress"] = progress
    
    with open(STATUS_FILE_PATH, 'w') as f:
        json.dump(status_data, f, indent=2)
    print(f"📝 Status: {status} - {message}")

class FaceTracker:
    """Track faces across frames using embedding similarity."""
    def __init__(self, similarity_threshold=0.65, max_disappeared=30, update_alpha=0.9):
        self.next_person_id = 1
        self.active_people = {}
        self.similarity_threshold = float(similarity_threshold)
        self.max_disappeared = int(max_disappeared)
        self.update_alpha = float(update_alpha)
        self.disappeared_frames = defaultdict(int)

    def _normalize(self, emb):
        emb = np.asarray(emb, dtype=np.float32).reshape(-1)
        norm = float(np.linalg.norm(emb) + 1e-12)
        return emb / norm

    def get_cosine_similarity(self, embedding1, embedding2):
        e1 = self._normalize(embedding1)
        e2 = self._normalize(embedding2)
        return float(np.dot(e1, e2))

    def _register(self, embedding, timestamp):
        person_id = self.next_person_id
        self.active_people[person_id] = {
            "embedding": self._normalize(embedding),
            "last_seen": float(timestamp),
        }
        self.disappeared_frames[person_id] = 0
        self.next_person_id += 1
        return person_id

    def _mark_disappeared(self):
        for person_id in list(self.active_people.keys()):
            self.disappeared_frames[person_id] += 1
            if self.disappeared_frames[person_id] > self.max_disappeared:
                del self.active_people[person_id]
                del self.disappeared_frames[person_id]

    def update(self, faces, timestamp):
        if not faces:
            self._mark_disappeared()
            return []

        if not self.active_people:
            assigned = []
            for face_data in faces:
                assigned.append(self._register(face_data["embedding"], timestamp))
            return assigned

        person_ids = list(self.active_people.keys())
        stored = np.stack([self.active_people[pid]["embedding"] for pid in person_ids], axis=0)

        new_embeddings = [self._normalize(f["embedding"]) for f in faces]
        new_mat = np.stack(new_embeddings, axis=0)

        sim = new_mat @ stored.T

        assigned_ids = [None] * len(faces)
        used_people = set()
        used_faces = set()

        candidates = []
        for i in range(sim.shape[0]):
            for j in range(sim.shape[1]):
                candidates.append((float(sim[i, j]), i, j))
        candidates.sort(reverse=True, key=lambda x: x[0])

        for s, i, j in candidates:
            if s < self.similarity_threshold:
                break
            if i in used_faces:
                continue
            pid = person_ids[j]
            if pid in used_people:
                continue

            assigned_ids[i] = pid
            used_faces.add(i)
            used_people.add(pid)

            old_emb = self.active_people[pid]["embedding"]
            new_emb = new_embeddings[i]
            updated = self.update_alpha * old_emb + (1.0 - self.update_alpha) * new_emb
            self.active_people[pid]["embedding"] = self._normalize(updated)
            self.active_people[pid]["last_seen"] = float(timestamp)
            self.disappeared_frames[pid] = 0

        for i, face_data in enumerate(faces):
            if assigned_ids[i] is None:
                assigned_ids[i] = self._register(face_data["embedding"], timestamp)

        for pid in list(self.active_people.keys()):
            if pid not in used_people:
                self.disappeared_frames[pid] += 1
                if self.disappeared_frames[pid] > self.max_disappeared:
                    del self.active_people[pid]
                    del self.disappeared_frames[pid]

        return assigned_ids


def create_unique_color(identifier):
    """Create a unique color based on the identifier."""
    hash_val = hash(identifier)
    r = (hash_val & 0xFF0000) >> 16
    g = (hash_val & 0x00FF00) >> 8
    b = hash_val & 0x0000FF
    return (b, g, r)


def get_emotion_color(emotion):
    """Return color based on emotion type."""
    emotion_colors = {
        "angry": (0, 0, 255),      # Red
        "disgust": (0, 128, 0),    # Dark Green
        "fear": (128, 0, 128),     # Purple
        "happy": (0, 255, 255),    # Yellow
        "sad": (255, 0, 0),        # Blue
        "surprise": (0, 165, 255), # Orange
        "neutral": (128, 128, 128) # Gray
    }
    return emotion_colors.get(emotion.lower(), (255, 255, 255))


print("✅ Helper functions and FaceTracker class defined")

## 6. Main Processing - Face Detection & Emotion Analysis

**This uses your exact working code:**
- **InsightFace** for face detection and tracking
- **DeepFace** for emotion analysis with 5-second smoothing window
- Prevents ID switching when faces turn slightly
- Emotion smoothing prevents rapid emotion changes

In [ ]:
# ========================================
# MAIN PROCESSING
# ========================================

from insightface.app import FaceAnalysis
from deepface import DeepFace

# Emotion smoothing settings
EMOTION_WINDOW_SECONDS = 5.0  # Time window to accumulate emotion scores
MIN_CONFIDENCE_THRESHOLD = 30.0  # Minimum confidence to consider an emotion

update_status("PROCESSING", "Starting face detection and emotion analysis...")

# Force CPU provider for processing
print("ℹ️ Using CPU for processing")
providers_to_use = ["CPUExecutionProvider"]

# Initialize InsightFace for face detection
app = FaceAnalysis(name="buffalo_s", providers=providers_to_use)
app.prepare(ctx_id=0, det_size=(640, 640))

# Open input video
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    update_status("FAILED", f"Input video not found: {INPUT_VIDEO_PATH}")
    raise FileNotFoundError(f"Video not found: {INPUT_VIDEO_PATH}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Create video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

print(f"Processing {total_frames} frames...")
print(f"Emotion smoothing: {EMOTION_WINDOW_SECONDS}s window")

# Initialize face tracker (lower threshold prevents ID switching)
tracker = FaceTracker(similarity_threshold=0.45, max_disappeared=50)

# Storage for results
all_emotion_detections = []
face_detections_for_emotion = {}  # {timestamp: [face_data]}

# Emotion history for smoothing: {person_id: [(timestamp, emotion_scores), ...]}
emotion_history = defaultdict(list)
stable_emotions = {}

frame_idx = 0

# PHASE 1: Face Detection with InsightFace
update_status("PROCESSING", "Phase 1/2: Detecting and tracking faces...")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_idx += 1
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    ts_key = round(float(timestamp), 2)

    # Detect faces with InsightFace
    try:
        faces = app.get(frame)
        
        face_items = []
        for f in faces:
            if getattr(f, "embedding", None) is None:
                continue
            
            x1, y1, x2, y2 = f.bbox
            x = max(0, int(x1))
            y = max(0, int(y1))
            w = max(0, int(x2 - x1))
            h = max(0, int(y2 - y1))
            
            if w <= 0 or h <= 0:
                continue
                
            face_items.append({
                "embedding": np.asarray(f.embedding, dtype=np.float32),
                "bbox": [x, y, w, h],
            })

        # Track faces across frames
        assigned_ids = tracker.update(face_items, timestamp)
        
        # Store face data for emotion analysis
        face_detections_for_emotion[ts_key] = []
        for face, person_id in zip(face_items, assigned_ids):
            if person_id is None:
                continue
            face_detections_for_emotion[ts_key].append({
                "person_id": f"person_{int(person_id)}",
                "bbox": face["bbox"],
            })

        # Progress logging
        if frame_idx % 30 == 0:
            progress = int((frame_idx / total_frames) * 50)  # 50% for phase 1
            ids_str = ", ".join([f"person_{pid}" for pid in assigned_ids]) if assigned_ids else "none"
            print(f"[Frame {frame_idx}/{total_frames}] Faces: {len(face_items)} | IDs: [{ids_str}]")
            update_status("PROCESSING", f"Face detection: {progress}% ({frame_idx}/{total_frames} frames)", progress)

    except Exception as e:
        print(f"Error processing frame {frame_idx}: {str(e)}")
        continue

cap.release()

# PHASE 2: Emotion Analysis with DeepFace
update_status("PROCESSING", "Phase 2/2: Analyzing emotions with DeepFace...")

cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
frame_idx = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_idx += 1
    current_ts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    ts_key = round(float(current_ts), 2)

    # Get face detections for this frame
    current_faces = face_detections_for_emotion.get(ts_key, [])

    # Process each detected face
    for face in current_faces:
        pid = face["person_id"]
        x, y, w, h = face["bbox"]

        # Extract face region
        face_roi = frame[y:y+h, x:x+w]
        
        if face_roi.size == 0:
            continue

        # Analyze emotion using DeepFace
        try:
            result = DeepFace.analyze(
                face_roi,
                actions=["emotion"],
                enforce_detection=False,
                silent=True
            )

            if isinstance(result, list):
                result = result[0]

            emotion_scores = result["emotion"]

            # Add to emotion history for smoothing
            emotion_history[pid].append((current_ts, emotion_scores))

            # Remove old entries outside the time window
            cutoff_time = current_ts - EMOTION_WINDOW_SECONDS
            emotion_history[pid] = [(ts, scores) for ts, scores in emotion_history[pid] if ts >= cutoff_time]

            # Calculate accumulated emotion scores
            accumulated_scores = defaultdict(float)
            for ts, scores in emotion_history[pid]:
                for emotion, score in scores.items():
                    accumulated_scores[emotion] += score

            # Find dominant emotion
            dominant_emotion = max(accumulated_scores, key=accumulated_scores.get)
            confidence = emotion_scores[dominant_emotion]

            # Store stable emotion
            stable_emotions[pid] = (dominant_emotion, confidence)

            # Save emotion detection
            emotion_detection = {
                "timestamp": ts_key,
                "person_id": pid,
                "coordinates_pixels": [int(x), int(y), int(w), int(h)],
                "emotion": dominant_emotion,
                "confidence": round(float(confidence), 2),
                "all_emotions": {k: round(float(v), 2) for k, v in emotion_scores.items()}
            }
            all_emotion_detections.append(emotion_detection)

            # Draw on frame
            person_color = create_unique_color(pid)
            emotion_color = get_emotion_color(dominant_emotion)

            # Draw face rectangle
            cv2.rectangle(frame, (x, y), (x + w, y + h), person_color, 2)

            # Draw person ID label
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            thickness = 2

            (text_w, text_h), _ = cv2.getTextSize(pid, font, font_scale, thickness)
            cv2.rectangle(frame, (x, y - text_h - 10), (x + text_w + 10, y), person_color, -1)
            cv2.putText(frame, pid, (x + 5, y - 5), font, font_scale, (255, 255, 255), thickness)

            # Draw emotion label
            label_emotion = f"{dominant_emotion} ({confidence:.0f}%)"
            (emo_w, emo_h), _ = cv2.getTextSize(label_emotion, font, font_scale, thickness)
            cv2.rectangle(frame, (x, y + h), (x + emo_w + 10, y + h + emo_h + 10), emotion_color, -1)
            cv2.putText(frame, label_emotion, (x + 5, y + h + emo_h + 5), font, font_scale, (255, 255, 255), thickness)

        except Exception as e:
            # If emotion detection fails, just draw the box
            person_color = create_unique_color(pid)
            cv2.rectangle(frame, (x, y), (x + w, y + h), person_color, 2)
            cv2.putText(frame, pid, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, person_color, 2)

    # Add frame counter
    cv2.putText(frame, f"Frame: {frame_idx}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    out.write(frame)

    # Progress logging
    if frame_idx % 30 == 0:
        progress = 50 + int((frame_idx / total_frames) * 50)  # 50-100% for phase 2
        print(f"[Frame {frame_idx}/{total_frames}] Emotions detected")
        update_status("PROCESSING", f"Emotion analysis: {progress}% ({frame_idx}/{total_frames} frames)", progress)

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"\n✓ Processing complete!")
print(f"  - Total emotion detections: {len(all_emotion_detections)}")
print(f"  - Output video: {OUTPUT_VIDEO_PATH}")
print(f"  - Output JSON: {OUTPUT_JSON_PATH}")

## 8. Save Results to Google Drive

In [ ]:
# ========================================
# GENERATE EMOTION SUMMARY & SAVE JSON
# ========================================
from collections import defaultdict
import json
import os

print("📊 Generating emotion summary...")

# Generate emotion summary from all detections
emotion_summary = defaultdict(lambda: {
    'frame_count': 0,
    'emotions': defaultdict(int),
    'frames': []
})

for detection in all_emotion_detections:
    person_id = detection['person_id']
    emotion = detection['emotion']
    timestamp = detection['timestamp']
    
    emotion_summary[person_id]['frame_count'] += 1
    emotion_summary[person_id]['emotions'][emotion] += 1
    emotion_summary[person_id]['frames'].append({
        'timestamp': timestamp,
        'emotion': emotion,
        'confidence': detection['confidence'],
        'coordinates': detection['coordinates_pixels']
    })

# Convert to regular dict for JSON serialization
emotion_summary = {k: {
    'frame_count': v['frame_count'],
    'emotions': dict(v['emotions']),
    'frames': v['frames']
} for k, v in emotion_summary.items()}

print(f"✅ Emotion summary generated for {len(emotion_summary)} person(s)")
for person_id, data in emotion_summary.items():
    print(f"  {person_id}: {data['frame_count']} frames")

# ========================================
# SAVE ALL RESULTS TO GOOGLE DRIVE
# ========================================

# Create output data structure that backend expects
output_data = {
    "summary_by_person": emotion_summary,
    "total_frames": frame_idx,
    "total_detections": len(all_emotion_detections),
    "detections": all_emotion_detections
}

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_JSON_PATH), exist_ok=True)

# Save JSON file
print(f"\n📝 Saving results to: {OUTPUT_JSON_PATH}")
try:
    with open(OUTPUT_JSON_PATH, 'w') as f:
        json.dump(output_data, f, indent=2)
    
    # Verify file was created
    if os.path.exists(OUTPUT_JSON_PATH):
        file_size = os.path.getsize(OUTPUT_JSON_PATH)
        print(f"✅ JSON file saved successfully!")
        print(f"   File size: {file_size / 1024:.2f} KB")
    else:
        print(f"❌ JSON file was not created!")
except Exception as e:
    print(f"❌ Error saving JSON: {e}")
    import traceback
    traceback.print_exc()

# Display emotion summary
print("\n📊 Emotion Summary:")
if emotion_summary:
    for person_id, data in emotion_summary.items():
        print(f"\n{person_id.upper()}:")
        print(f"  Total frames: {data['frame_count']}")
        print(f"  Emotions:")
        for emotion, count in sorted(data['emotions'].items(), key=lambda x: x[1], reverse=True):
            percentage = (count / data['frame_count']) * 100
            print(f"    {emotion}: {count} frames ({percentage:.1f}%)")
else:
    print("  ⚠️ No faces detected in video!")

# Final status update
update_status("DONE", f"Processing complete! Detected {len(emotion_summary)} person(s) across {frame_idx} frames")

print("\n🎉 ALL DONE! Refresh your web application to see results.")

## 9. Verify Output Files

In [ ]:
# Verify all output files exist
print("=" * 50)
print("📋 VERIFYING OUTPUT FILES")
print("=" * 50)

files_to_check = [
    (OUTPUT_VIDEO_PATH, "Output video"),
    (OUTPUT_JSON_PATH, "Emotion JSON"),
    (STATUS_FILE_PATH, "Status file")
]

all_exist = True
for file_path, description in files_to_check:
    exists = os.path.exists(file_path)
    status = "✅" if exists else "❌"
    size = os.path.getsize(file_path) if exists else 0
    print(f"\n{status} {description}:")
    print(f"   Path: {file_path}")
    if exists:
        print(f"   Size: {size / 1024:.2f} KB ({size / (1024*1024):.2f} MB)")
    else:
        print(f"   ⚠️ FILE NOT FOUND!")
    all_exist = all_exist and exists

print("\n" + "=" * 50)
if all_exist:
    print("✅ ALL FILES CREATED SUCCESSFULLY!")
    print("\n👉 NEXT STEPS:")
    print(f"   1. Go to your web app: http://localhost:3000")
    print(f"   2. Navigate to Results page for job: {JOB_ID}")
    print(f"   3. Or go directly to: http://localhost:3000/results/{JOB_ID}")
else:
    print("❌ SOME FILES ARE MISSING!")
    print("\n⚠️ Check the processing output above for errors.")
    print("   - Make sure the video file exists in input_videos folder")
    print("   - Make sure faces were detected in the video")
print("=" * 50)

---

## 📝 Notes

### What This Notebook Does:
- ✅ Reads video from Google Drive
- ✅ Detects faces with InsightFace (GPU)
- ✅ Detects emotions with DeepFace
- ✅ Annotates video frames
- ✅ Writes output video to Google Drive
- ✅ Writes emotion JSON to Google Drive
- ✅ Updates status for live monitoring

### What This Notebook Does NOT Do:
- ❌ Automatically trigger on video upload
- ❌ Communicate directly with Django backend
- ❌ Send notifications or webhooks

### Workflow:
1. Upload video via web app
2. Web app saves to Google Drive
3. **YOU MANUALLY RUN THIS NOTEBOOK** (Run All)
4. Notebook processes video with GPU
5. Notebook writes results to Google Drive
6. Web app reads results and displays them

### Tips:
- Use GPU runtime (Runtime > Change runtime type > GPU)
- Make sure Google Drive is mounted
- Update JOB_ID in configuration cell
- Check status updates in web app while processing
- If processing fails, check error messages above

---